In [0]:
%pip install -U \
langchain \
langchain-core \
langchain-community \
langchain-text-splitters \
databricks-langchain \
databricks-vectorsearch \
mlflow
# Restart
dbutils.library.restartPython()
# Check Version
import langchain
print(langchain.__version__)

In [0]:
from databricks_langchain import DatabricksVectorSearch
from databricks_langchain.chat_models import ChatDatabricks

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

import mlflow
from mlflow.models import infer_signature

# ----------------------------
# CONFIG
# ----------------------------
VECTOR_INDEX = "workspace.default.studebaker_vector_search"
LLM_ENDPOINT = "databricks-llama-4-maverick"

# ----------------------------
# VECTOR SEARCH
# ----------------------------
vector_store = DatabricksVectorSearch(index_name=VECTOR_INDEX)

retriever = vector_store.as_retriever(search_kwargs={"k": 5})

# ----------------------------
# LLM
# ----------------------------
llm = ChatDatabricks(
    endpoint=LLM_ENDPOINT, temperature=0.1, max_tokens=400
)

# ----------------------------
# PROMPT
# ----------------------------
prompt = ChatPromptTemplate.from_template(
    """You are an expert Studebaker mechanic that responds with repair directions and part numbers for impacted components.
Some pieces of context may be irrelevant, in which case you should not use them to form the answer.
Use the following context to answer the question.
Cite documents when utilized.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context:
{context}

Question:
{question}
"""
)


# ----------------------------
# FORMAT DOCUMENTS
# ----------------------------
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# ----------------------------
# LCEL RAG CHAIN
# ----------------------------
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# ----------------------------
# RUN QUERY
# ----------------------------
questions = [
    "How do I repair the windshield wiper motor on a Studebaker truck?",
    "How do I replace a front brake shoe on a studebaker truck?"
]
for i, q in enumerate(questions):
    response = rag_chain.invoke(
        input=q
    )
    print(response, "******\n\n\n")
    # Demo Save off for Ground Truth
    # with open(f"{i}_ground_truth.txt", "w") as file:
    #     file.write(response)

In [0]:
import pandas as pd

def load_groundtruth():
    # ----------------------------
    # EXAMPLE EVALUATION DATA
    # ----------------------------
    # This is your "ground truth" set for evaluation
    gt_files = [
        "windshield_repair_ground_truth.txt",
        "brake_repair_ground_truth.txt"
    ]
    # Build out Ground Truth somewhat dynamically.
    eval_data = []
    for gt_file, question in zip(gt_files, questions):
        with open(gt_file) as gt:
            gt_data = gt.read()
            eval_data.append(
                {
                    "question": question,
                    "answer": gt_data
                }
            )
            print(f"Length of Truth: {len(gt_data)}")

    eval_df = pd.DataFrame(eval_data)
    return eval_df

# Load Ground Truth
groundtruth_df = load_groundtruth()
display(groundtruth_df)

In [0]:
import pandas as pd
import mlflow
from mlflow.metrics import precision_at_k, recall_at_k, ndcg_at_k
from mlflow.deployments import set_deployments_target
from mlflow.metrics.genai import relevance
from mlflow.metrics import latency


# Load Fresh Ground Truth
groundtruth_df = load_groundtruth()

# ----------------------------
# START MLflow RUN
# ----------------------------
with mlflow.start_run(run_name="studebaker_rag_chain") as run:

    # Log chain config
    mlflow.log_param("vector_index", VECTOR_INDEX)
    mlflow.log_param("llm_endpoint", LLM_ENDPOINT)
    mlflow.log_param("retriever_k", 5)
    mlflow.log_param("temperature", 0.1)

    # Evaluate, These should be replaced wtih better Evaluations.
    predictions = []
    references = []
    for item in eval_data:
        result = rag_chain.invoke(item["question"])
        predictions.append(result)
        references.append(item["answer"])

    # Simple metric: exact match
    exact_matches = [p.strip().lower() == r.strip().lower() for p, r in zip(predictions, references)]
    accuracy = sum(exact_matches) / len(exact_matches)

    mlflow.log_metric("exact_match_accuracy", accuracy)

    # Optionally log the full predictions
    for i, (q, p, r) in enumerate(zip([d["question"] for d in eval_data], predictions, references)):
        mlflow.log_text(f"Q: {q}\nPred: {p}\nRef: {r}", f"prediction_{i}.txt")

    # Write chain definition to file (models-from-code pattern required by LangChain v1+)
    chain_code_path = "/tmp/rag_chain_code.py"
    with open(chain_code_path, "w") as f:
        f.write('''import mlflow
from databricks_langchain import DatabricksVectorSearch
from databricks_langchain.chat_models import ChatDatabricks
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

VECTOR_INDEX = "workspace.default.studebaker_vector_search"
LLM_ENDPOINT = "databricks-llama-4-maverick"

vector_store = DatabricksVectorSearch(index_name=VECTOR_INDEX)
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

llm = ChatDatabricks(endpoint=LLM_ENDPOINT, temperature=0.1, max_tokens=400)

prompt = ChatPromptTemplate.from_template(
    """You are an expert Studebaker mechanic that responds with repair directions and part numbers for impacted components.
Some pieces of context may be irrelevant, in which case you should not use them to form the answer.
Use the following context to answer the question.
Cite documents when utilized.
If you don\'t know the answer, just say that you don\'t know, don\'t try to make up an answer.

Context:
{context}

Question:
{question}
"""
)

def format_docs(docs):
    return "\\n\\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

mlflow.models.set_model(rag_chain)
''')

    # Log the rag_chain using MLflow's LangChain integration
    mlflow.langchain.log_model(
        lc_model=chain_code_path,
        artifact_path="rag_chain_model",
    )

    print(f"MLflow run completed. Run ID: {run.info.run_id}, Accuracy: {accuracy}")


